# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. It demonstrates how to work with data whose metadata and schema are described using Croissant, and shows how to load, filter, and analyze the dataset in Python.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their `@id`, fields, and columns. Here, we print the metadata for each record set, their fields and columns to understand the data structure.


In [ ]:
# List all record sets
record_sets = metadata.record_sets
print("Available record sets (with their @id):")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, description: {rs.description}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    if hasattr(rs, 'columns') and rs.columns is not None:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - @id: {col.id}, name: {col.name}")
    print("\n")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

For this example, we'll extract the main protein data record set, which typically contains protein abundance and related measures.

If you aren't sure which record set holds the table data, see the output above and choose the relevant `@id`.


In [ ]:
# Select the main protein data record set by its @id (update as needed based on the listing above).
# For this dataset, we expect one main data table, likely with a name like "Protein Table" or "Proteins".
# Replace below with the @id from the overview above, e.g., '@id': 'https://api.app.sen.science/frontiers/7154140/a43f7d6e-example'
main_record_set_id = None
for rs in record_sets:
    # Select the first non-empty data table as an example
    if hasattr(rs, 'fields') and rs.fields:
        main_record_set_id = rs.id
        break

if main_record_set_id is None:
    raise ValueError("Main protein record set could not be found. Check the record set listing above.")

print(f"Selected record set @id: {main_record_set_id}\n")

# You can add more record set IDs if relevant.
selected_record_sets = [main_record_set_id]
dataframes = {}

for record_set_id in selected_record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Loaded columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing steps, such as filtering records based on a criterion, normalizing numeric fields, and grouping data.

Below, we select a numeric field (such as 'abundance', 'MW', etc.) by its `@id` or column name and demonstrate filtering and normalization.


In [ ]:
# List the columns to decide which numeric field to analyze
df = dataframes[main_record_set_id]
print("Available fields for analysis (column names):")
print(df.columns.tolist())

# Update the field name below as needed (e.g., 'abundance' or 'MW' or another measure)
# If the data contains normalized abundance, peptide count, MW, or coverage fields, pick one
# Here we demonstrate with a typical field name encountered in proteomics
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first numeric field found
else:
    # Try known likely fields if not autodetected
    for fn in ['MW', 'Molecular Weight', 'abundance', 'Abundance', 'coverage', 'Peptide Count', 'peptide_count']:
        if fn in df.columns:
            numeric_field = fn
            break
    else:
        raise ValueError('No obvious numeric field found, please specify one.')

print(f"Using numeric field: {numeric_field}\n")

# Some columns may be objects/strings that represent numbers, try to convert if needed
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter records with numeric_field > threshold
threshold = df[numeric_field].quantile(0.8)  # For demonstration, top 20% values
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a likely categorical field (e.g., description or accession/ID)
group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
# Example: group by 'description' if it exists
for group_field in ['Peptide Description', 'Protein', 'description', 'accession', 'ID']:
    if group_field in df.columns:
        break
else:
    group_field = group_candidates[0] if group_candidates else None

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Below, we visualize the distribution of the numeric field and, if grouping information is available, plot group means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Plot histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Optional: boxplot by group
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    # Plot only the top 10 groups by number of entries for cleaner display
    group_counts = df[group_field].value_counts().nlargest(10).index
    sns.boxplot(y=numeric_field, x=group_field, data=df[df[group_field].isin(group_counts)])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² mass spectrometry dataset, examined its structure, selected key record sets and fields by their `@id`, filtered and normalized the data, and visualized distributions. This approach can be extended to more detailed downstream proteomics analysis, group comparisons, or integration with external databases.

Always refer to field and record set `@id`s when programmatically processing Croissant-based datasets for reproducibility and interoperability.